In [ ]:
## Combines all LipoGrid MALDI-MSI runs (first 2 plates: positive mode only; last 2: pos + neg mode)
## and aligns them with the matching Xenium runs to build a per-cell lipid profile. Memory heavy.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy.spatial import distance

sys.path.append(str(Path.cwd().parent / "scripts"))
from style import set_default_style

set_default_style()

In [ ]:
# one huge adata object from FOCUS's pipeline with all 4 MALDI-MSI runs, first 2 in pos mode, 2 last ones in pos and neg mode
adata_MSI_all = ad.read_h5ad('../data/lipogrid/pilot/MALDI_MSI/d_lipogrid/merged/preprocessing/MSI_merged_processed.h5ad') # the processed MSI data from slide 1 (also used for alignment)
adata_MSI_all 

In [ ]:
adata_MSI_all.var['lipid_annotation'].value_counts() # 655 lipids annotated from the 7420 mz values

In [ ]:
## check if own assignment of lipids to m/z agrees with FOCUS's annotation
# (760.340 - x) / 760.340 = 10 ppm = 10e-6; use 15 ppm around each mz to assign a lipid class
mz_to_lipids_pos = pd.read_csv('../data/metadata/MS1_database_POS_JI_JD_XS_NR_Washed.csv', sep = ',')
mz_to_lipids_neg = pd.read_csv('../data/metadata/MS1_database_NEG_JI_JD_XS_NR_Washed.csv', sep = ',')

## remove all artificial lipids they use as standards (with deuterium) as we don't expect them in our dataset
mz_to_lipids_pos.columns = ['lipid', 'mz']
mz_to_lipids_neg.columns = ['lipid', 'mz']
# remove ; from mz
mz_to_lipids_pos['mz'] = mz_to_lipids_pos['mz'].str.replace(';', '').astype(float)
mz_to_lipids_neg['mz'] = mz_to_lipids_neg['mz'].str.replace(';', '').astype(float)

mz_to_lipids_pos = mz_to_lipids_pos[~mz_to_lipids_pos['lipid'].str.contains(' d')]  # Remove deuterated lipids
mz_to_lipids_neg = mz_to_lipids_neg[~mz_to_lipids_neg['lipid'].str.contains(' d')]  # Remove deuterated lipids

# Remove all the lipids in mz_to_lipids that have [M+Na]+ or [M+K]+ but not a / as these ions should have been depleted during the washes in the MALDI procedure
mask = (
    mz_to_lipids_pos['lipid'].str.contains(r'\[M\+Na\]\+|\[M\+K\]\+', regex=True)
)
mz_to_lipids_pos = mz_to_lipids_pos[~mask]

mask = (
    mz_to_lipids_neg['lipid'].str.contains(r'\[M\+Na\]\+|\[M\+K\]\+', regex=True)
)
mz_to_lipids_neg = mz_to_lipids_neg[~mask]
## split in mz's measured in positive mode and those measured in negative mode

mz = adata_MSI_all.var['mz'].astype(float)

start = mz - mz*10e-6
end = mz + mz*10e-6
mz_reference = pd.DataFrame(mz, columns=['mz'])
mz_reference['start'] = start
mz_reference['end'] = end
mz_reference['mode'] = adata_MSI_all.var['mz_mode'].values
mz_reference

## add anotated lipids from the mz_to_lipids list if they mz values falls within the range of the start and end values of the mz_reference
# Add a column for annotation, default None
mz_reference['lipid_annotation'] = None

for idx, row in mz_reference.iterrows():
    # Find all lipids whose mz falls within the start-end range for mz_to_lipids_pos if mz_reference mode is pos
    if row['mode'] == 'pos':
        matches = mz_to_lipids_pos[(mz_to_lipids_pos['mz'] >= row['start']) & (mz_to_lipids_pos['mz'] <= row['end'])]
        if not matches.empty:
         # If multiple matches, join their names
            mz_reference.at[idx, 'lipid_annotation'] = ';'.join(matches['lipid'].astype(str))  
        else:
         mz_reference.at[idx, 'lipid_annotation'] = 'Unannotated'
    elif row['mode'] == 'neg':
        matches = mz_to_lipids_neg[(mz_to_lipids_neg['mz'] >= row['start']) & (mz_to_lipids_neg['mz'] <= row['end'])]
        if not matches.empty:
         # If multiple matches, join their names
            mz_reference.at[idx, 'lipid_annotation'] = ';'.join(matches['lipid'].astype(str))             
        else:
         mz_reference.at[idx, 'lipid_annotation'] = 'Unannotated'
        
adata_MSI_all.var['lipid_annotation_self'] = mz_reference['lipid_annotation'].values

# Show annotated mz_reference
adata_MSI_all.var['lipid_annotation_self'].value_counts() # 423 peaks are annotated without Na or K adducts! (this is before intensity filtering)!

## aligment MALDI-MSI and Xenium runs to obtain for each cell detected in Xenium the lipid profile from the MALDI-MSI data

In [ ]:
## Allinged files from FOCUS
adata_Xenium_all = ad.read_h5ad('../data/lipogrid/pilot/MALDI_MSI/d_lipogrid/merged/alignment/Xenium_merged_processed_aligned.h5ad')
adata_Xenium_all

In [ ]:
## split both MSI and Xenium data per slide
SAMPLE_IDS = ["20250606_RUN01_01", "20250606_RUN01_02", "20251119_RUN02_01", "20251205_RUN03_01"]

adata_MSI_slides = [adata_MSI_all[adata_MSI_all.obs['sample_id'] == sid].copy() for sid in SAMPLE_IDS]
adata_Xenium_slides = [adata_Xenium_all[adata_Xenium_all.obs['sample_id'] == sid].copy() for sid in SAMPLE_IDS]

In [ ]:
# keep only Xenium cells whose MSI-aligned coordinates fall inside that slide's MSI raster
for i, (adata_msi, adata_xenium) in enumerate(zip(adata_MSI_slides, adata_Xenium_slides)):
    max_msi = adata_msi.obsm['raster_coordinates'].max(axis=(0, 1))
    coords = adata_xenium.obsm['MSI_spatial']
    mask = (coords[..., 0] >= 0) & (coords[..., 0] < max_msi[0]) & (coords[..., 1] >= 0) & (coords[..., 1] < max_msi[1])
    adata_Xenium_slides[i] = adata_xenium[mask].copy()

adata_Xenium_slides

In [ ]:
# Generate single-cell MSI profiles: for each Xenium cell, take the inverse-distance-weighted
# mean of its 4 nearest MSI spots.
msi_cell_parts = []
for adata_msi, adata_xenium in zip(adata_MSI_slides, adata_Xenium_slides):
    xenium_coords = adata_xenium.obsm['MSI_spatial']                     # (n_cells, 2)
    msi_spot_centers = adata_msi.obsm['raster_coordinates'].mean(axis=1)  # (n_spots, 2)

    dists = distance.cdist(xenium_coords, msi_spot_centers)  # (n_cells, n_spots)
    closest4_idx = np.argpartition(dists, 4, axis=1)[:, :4]
    row_idx = np.arange(dists.shape[0])[:, None]
    closest4_dists = dists[row_idx, closest4_idx]

    msi_agg = adata_msi.X  # (n_spots, n_features)
    aggregated = np.array([
        np.sum(msi_agg[idxs] * (1 / (d + 1e-8))[:, None], axis=0) / np.sum(1 / (d + 1e-8))
        for idxs, d in zip(closest4_idx, closest4_dists)
    ])  # (n_cells, n_features)

    msi_cell_parts.append(ad.AnnData(X=aggregated, obs=adata_xenium.obs.copy(), var=adata_msi.var.copy()))

msi_int_cell = ad.concat(msi_cell_parts, axis=0, join='outer')
msi_int_cell.var = adata_MSI_slides[-1].var.copy()

# Obtain the mz profile of the matrix and substract this from the cell profile

In [ ]:
# Background ("matrix") MSI spots: those not among the 12 nearest neighbours of any Xenium cell
msi_matrix_parts = []
for adata_msi, adata_xenium in zip(adata_MSI_slides, adata_Xenium_slides):
    adata_msi.obs['index'] = range(adata_msi.n_obs)

    xenium_coords = adata_xenium.obsm['MSI_spatial']
    msi_spot_centers = adata_msi.obsm['raster_coordinates'].mean(axis=1)

    dists = distance.cdist(xenium_coords, msi_spot_centers)
    closest12_idx = np.argpartition(dists, 12, axis=1)[:, :12]
    spots_with_cell = np.unique(closest12_idx.flatten())

    background = adata_msi[~adata_msi.obs['index'].isin(spots_with_cell)].copy()
    background.obsm.clear()
    background.layers.clear()
    msi_matrix_parts.append(background)

msi_int_matrix = ad.concat(msi_matrix_parts, axis=0, join='outer')
msi_int_matrix.var = adata_MSI_slides[-1].var
msi_int_matrix

## split filters in pos and negative for MZs

## postive mode was measured in all 4 samples

In [ ]:
## fiter in anndate vars to only retain mz_mode pos
msi_int_cell_pos = msi_int_cell.copy()
msi_int_cell_pos = msi_int_cell_pos[:, msi_int_cell_pos.var['mz_mode'] == 'pos']

msi_int_matrix_pos = msi_int_matrix.copy()
msi_int_matrix_pos = msi_int_matrix_pos[:, msi_int_matrix_pos.var['mz_mode'] == 'pos']
msi_int_cell_pos, msi_int_matrix_pos

In [ ]:
## calculate average profiles again for pos only
average_msi_profile_cells_pos = msi_int_cell_pos.X.mean(axis=0)
average_msi_profile_log_cap_cells_pos = np.log2(average_msi_profile_cells_pos+1).clip(max=10)

average_msi_profile_matrix_pos = msi_int_matrix_pos.X.mean(axis=0)
average_msi_profile_log_cap_mat_pos = np.log2(average_msi_profile_matrix_pos+1).clip(max=10)  # add 1 to avoid log2(0) and cap the values to avoid extreme values



In [ ]:
## histogram of the average msi profile
plt.figure(figsize=(8, 5))
plt.hist(average_msi_profile_log_cap_cells_pos, bins=100, color='blue', alpha=0.7)
plt.title('Average MSI Profile Cells (pos mode)')
plt.xlabel('Intensity')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(average_msi_profile_log_cap_mat_pos, bins=100, color='blue', alpha=0.7)
plt.title('Average MSI Profile Matrix (pos mode)')
plt.xlabel('Intensity')
plt.ylabel('Frequency')
plt.show()

## correlation plot between the average msi profile of the cells and the average msi profile of the filtered coordinates
plt.figure(figsize=(8, 5))
plt.scatter(average_msi_profile_log_cap_cells_pos, average_msi_profile_log_cap_mat_pos, s=1, c='blue', alpha=0.7)
plt.title('Correlation between Average MSI Profiles (pos mode)')
plt.xlabel('Average MSI Profile Cells (log2)')
plt.ylabel('Average MSI Profile Matrix (log2)')
plt.xlim(0, 10.1)
plt.ylim(0, 10.1)
plt.plot([0, 10], [0, 10], color='red', linestyle='--')  # Add a diagonal line for reference
plt.grid()
plt.show()

## Negative mode only for sample 3 and 4

In [ ]:
## fiter in anndate vars to only retain mz_mode neg AND remove S1 and S2 from the analyses
msi_int_cell_neg = msi_int_cell.copy()
msi_int_cell_neg = msi_int_cell_neg[:, msi_int_cell_neg.var['mz_mode'] == 'neg']
msi_int_cell_neg = msi_int_cell_neg[~msi_int_cell_neg.obs['sample_id'].str.contains('RUN01')]

msi_int_matrix_neg = msi_int_matrix.copy()
msi_int_matrix_neg = msi_int_matrix_neg[:, msi_int_matrix_neg.var['mz_mode'] == 'neg']
msi_int_matrix_neg = msi_int_matrix_neg[~msi_int_matrix_neg.obs['sample_id'].str.contains('RUN01')]
msi_int_cell_neg, msi_int_matrix_neg

In [ ]:
## calculate average profiles again for neg only
average_msi_profile_cells_neg = msi_int_cell_neg.X.mean(axis=0)
average_msi_profile_log_cap_cells_neg = np.log2(average_msi_profile_cells_neg+1).clip(max=10)

average_msi_profile_matrix_neg = msi_int_matrix_neg.X.mean(axis=0)
average_msi_profile_log_cap_mat_neg = np.log2(average_msi_profile_matrix_neg+1).clip(max=10)  

In [ ]:
## histogram of the average msi profile neg
plt.figure(figsize=(8, 5))
plt.hist(average_msi_profile_log_cap_cells_neg, bins=100, color='blue', alpha=0.7)
plt.title('Average MSI Profile Cells neg mode')
plt.xlabel('Intensity')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(average_msi_profile_log_cap_mat_neg, bins=100, color='blue', alpha=0.7)
plt.title('Average MSI Profile Matrix neg mode')
plt.xlabel('Intensity')
plt.ylabel('Frequency')
plt.show()

## correlation plot between the average msi profile of the cells and the average msi profile of the filtered coordinates
plt.figure(figsize=(8, 5))
plt.scatter(average_msi_profile_log_cap_cells_neg, average_msi_profile_log_cap_mat_neg, s=1, c='blue', alpha=0.7)
plt.title('Correlation between Average MSI Profiles neg mode')
plt.xlabel('Average MSI Profile Cells (log2)')
plt.ylabel('Average MSI Profile Matrix (log2)')
plt.xlim(0, 10.1)
plt.ylim(0, 10.1)
plt.plot([0, 10], [0, 10], color='red', linestyle='--')  # Add a diagonal line for reference
plt.grid()
plt.show()

## filtering the mz values based on enrichment over background

In [ ]:
fold_msi_profile_cells_over_mat_pos = (average_msi_profile_cells_pos+1)/(average_msi_profile_matrix_pos +1)
fold_msi_profile_cells_over_mat_neg = (average_msi_profile_cells_neg+1)/(average_msi_profile_matrix_neg +1)
fold_msi_profile_cells_over_mat_pos, fold_msi_profile_cells_over_mat_neg

In [ ]:
## add vars to anndata object with average signal in cells, in matrix and fold change. 
## These values will be used to filter out mz that are likely just stemming from the matrix and not from biomolecules
msi_int_cell.var['fold_change_cellVmat'] = np.concatenate([fold_msi_profile_cells_over_mat_pos, fold_msi_profile_cells_over_mat_neg], axis=0)
msi_int_cell.var['avg_intensity_cells'] = np.concatenate([average_msi_profile_cells_pos, average_msi_profile_cells_neg], axis=0)
msi_int_cell.var['avg_intensity_matrix'] = np.concatenate([average_msi_profile_matrix_pos, average_msi_profile_matrix_neg], axis=0)
## save anndata object

msi_int_cell.write('../data/lipogrid/pilot/analysis/final_4_runs/msi_int_cells.h5ad')
msi_int_cell.var